In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import tensorflow as tf
import keras
import numpy as np
from keras.models import Sequential
from keras.layers import Dropout,Flatten,Conv2D,MaxPooling2D,Dense,GlobalAveragePooling2D,BatchNormalization
from keras.regularizers import Regularizer
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import ModelCheckpoint

In [2]:
import tensorflow_datasets as tfds
import tensorflow as tf


### Step 1:Load Dataset

In [6]:
(train_data,valid_data),info=tfds.load('oxford_flowers102',split=['train','validation'],as_supervised=True,with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]
Dl Completed...:  67%|██████▋   | 2/3 [1:27:38<43:49, 2629.29s/ url]
Extraction completed...: 0 file [1:30:43, ? file/s]
Dl Completed...:  67%|██████▋   | 2/3 [1:30:43<45:21, 2721.99s/ url]


ChunkedEncodingError: ('Connection broken: IncompleteRead(128433912 bytes read, 216428597 more expected)', IncompleteRead(128433912 bytes read, 216428597 more expected))

In [ ]:
num_classes=info.features['label'].num_classes

### step 2:Preprocessing the data


In [ ]:
def preprocess(x,y):
    x=tf.image.resize(x,(227,227))
    x=x/255.0
    y=tf.one_hot(y,num_classes)
    return x,y



### step 3:make a data pipeline

In [ ]:
train_ds=train_data.map(preprocess).shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE)
valid_ds=valid_data.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

### step 4:Build alexnet architecture


In [1]:
model=Sequential()

## first convolution block
model.add(Conv2D(96,kernel_size=(11,11),strides=(4,4),activation='relu',input_shape=(227,227,3),padding='valid'))
model.add(MaxPooling2D((3,3),strides=(2,2),padding='valid'))
model.add(BatchNormalization())

## second convolution block
model.add(Conv2D(256,(5,5),strides=(1,1),activation='relu',padding='same'))
model.add(MaxPooling2D((3,3),strides=(2,2),padding='valid'))
model.add(BatchNormalization())

## Third Convolution block
model.add(Conv2D(384,(3,3),strides=(1,1),activation='relu',padding='valid'))
model.add(BatchNormalization())
## Fourth Convolution block
model.add(Conv2D(384,(3,3),strides=(1,1),activation='relu',padding='valid'))
model.add(BatchNormalization())
## Fifth Convolution block
model.add(Conv2D(256,(3,3),strides=(1,1),activation='relu',padding='valid'))
model.add(MaxPooling2D((3,3),strides=(2,2),padding='valid'))
model.add(BatchNormalization())

model.add(Flatten())
model.add(Dense(4096,activation='relu'))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(4096,activation='relu'))
model.add(Dropout(0.5))
model.add(BatchNormalization())
model.add(Dense(num_classes,activation='softmax'))


model.summary()


NameError: name 'Sequential' is not defined

### Step 6:Compile model

In [ ]:
loss='sparse_categorical_crossentropy'
adam=tf.keras.optimizers.Adam(learning_rate=0.001)


model.compile(optimizer=adam,loss=loss,metrics=['accuracy'])

### step 7:Train Model

In [ ]:
history=model.fit(train_ds,epochs=5,validation_data=valid_ds)